# Clean & Split — CC1-Only Design

Input: `data/processed/all_cases_labeled.csv` (merged + labeled, all 4 cases).

**Scope of this notebook: cleaning + splitting only.** No windowing, PCA, or model
training here — that's the next notebook. Every step below was decided from a fresh
analysis of the actual data on disk today, not inherited from any prior
`preprocessing*.ipynb`.

**What this notebook does:**
1. Undoes a scaling bug baked into the input file (global StandardScaler fit on
   `single_case1`, leaking into every other case).
2. Deduplicates rows — the raw data has a real, verified duplication artifact.
3. Marks per-container time gaps (metadata for the next notebook's windowing step).
4. Rescopes anomaly labels to drop `delay`/`loss` faults, which this module's
   CPU/memory-only feature set cannot detect.
5. Saves the fully cleaned dataset (all 4 cases).
6. Splits `complex_case1` (CC1) by time into train/val/test; keeps `single_case1`,
   `single_case2`, `complex_case2` intact as separate drift-evaluation sets.
7. Fits scaling on CC1-train only and applies it everywhere (no leakage).

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, RobustScaler
import joblib, os

pd.set_option('display.width', 200)

BASE             = r'c:\Users\jthar\Documents\Claude\Projects\module3\module3'
IN_PATH          = os.path.join(BASE, 'data', 'processed', 'all_cases_labeled.csv')
SC1_LABELED_PATH = os.path.join(BASE, 'data', 'labeled', 'single_case1_labeled.csv')
OUT_DIR          = os.path.join(BASE, 'data', 'processed')
MODEL_DIR        = os.path.join(BASE, 'models')
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

FEATURE_COLS = [
    'container_cpu_usage_seconds_rate',
    'container_cpu_system_seconds_rate',
    'container_cpu_user_seconds_rate',
    'container_memory_usage_bytes',
    'container_memory_working_set_bytes',
    'container_memory_rss',
    'container_memory_cache',
]

EXCLUDED_FAULT_TYPES = ['delay', 'loss']   # network-layer faults, invisible to CPU/memory container metrics
GAP_THRESH_SEC       = 30                  # > 2x the nominal 15s sampling interval = real collection gap
TRAIN_FRAC           = 0.70
VAL_FRAC             = 0.80                # cumulative; val = [TRAIN_FRAC, VAL_FRAC)
CLIP                 = 20.0                # safety clip after scaling

print('Paths and constants configured.')

Paths and constants configured.


## Step 0 — Recover unbiased features

`merge_and_normalize.ipynb` fit a `StandardScaler` on **single_case1**'s normal rows
and applied it to all 4 cases before saving `all_cases_labeled.csv`. For a design
that trains exclusively on **complex_case1**, that bakes single_case1's scale into
what should be CC1's own definition of "normal" — a leakage/mis-calibration bug, not
a modeling choice.

Fix: re-fit that exact scaler from the untouched `data/labeled/single_case1_labeled.csv`
(same file it was originally fit on) and invert it, recovering the original
per-container min-max values. This is done via clean linear algebra
(`x_orig = x_scaled * scale_ + mean_`) — verified below to be exact, not approximate.

In [2]:
print('Loading merged/labeled dataset ...')
df = pd.read_csv(IN_PATH, low_memory=False)
print(f'  {len(df):,} rows x {df.shape[1]} columns  |  cases: {df["case"].unique().tolist()}')

sc1_labeled = pd.read_csv(SC1_LABELED_PATH, low_memory=False)
legacy_scaler = StandardScaler().fit(sc1_labeled.loc[sc1_labeled['label'] == 0, FEATURE_COLS])

recovered = df[FEATURE_COLS].values * legacy_scaler.scale_ + legacy_scaler.mean_
df[FEATURE_COLS] = recovered

# Sanity check: recovered single_case1 normals must match the original labeled file exactly
check_a = df.loc[(df['case'] == 'single_case1') & (df['label'] == 0), FEATURE_COLS].values
check_b = sc1_labeled.loc[sc1_labeled['label'] == 0, FEATURE_COLS].values
print(f'  Inversion check (max abs diff, should be ~0): {np.abs(check_a - check_b).max():.2e}')

Loading merged/labeled dataset ...
  533,230 rows x 12 columns  |  cases: ['single_case1', 'single_case2', 'complex_case1', 'complex_case2']
  Inversion check (max abs diff, should be ~0): 2.22e-16


## Step 1 — Deduplicate

Verified directly on `data/merged/*.csv`: every ~2 hours, on the hour, **every
container** gets a burst of 64–2187 fully/near-identical duplicate rows at the exact
same timestamp. This is a periodic data-collection artifact (confirmed to hit all 27
containers simultaneously, unrelated to any fault window) — 15–30% of rows per case
are duplicates. `EDA/preprocessing.ipynb` never deduplicated before computing labels
and CPU rates, so this is still present in the input file and must be cleaned now.

In [3]:
print('Deduplicating (case, cmdb_id, timestamp) ...')
before = len(df)
df = df.sort_values(['case', 'cmdb_id', 'timestamp']).reset_index(drop=True)
df = df.drop_duplicates(subset=['case', 'cmdb_id', 'timestamp'], keep='first').reset_index(drop=True)
after = len(df)
print(f'  {before:,} -> {after:,} rows  (removed {before - after:,} duplicate readings)')
print()
for case, g in df.groupby('case'):
    print(f'  {case:15s}: {len(g):>7,} rows')

Deduplicating (case, cmdb_id, timestamp) ...
  533,230 -> 400,896 rows  (removed 132,334 duplicate readings)

  complex_case1  : 223,803 rows
  complex_case2  :  77,760 rows
  single_case1   :  60,480 rows
  single_case2   :  38,853 rows


## Step 2 — Mark per-container time gaps

Sampling is nominally every 15s. After dedup, mark rows that start a real gap
(> 30s since the previous reading for that container) so the *next* notebook's
sliding-window step can skip windows that span a gap. Not resolved here — just
carried forward as metadata (`time_diff`, `is_gap`).

In [4]:
print('Marking per-container time gaps ...')
df['time_diff'] = df.groupby(['case', 'cmdb_id'])['timestamp'].diff()
df['time_diff'] = df['time_diff'].fillna(15.0)          # first row per container: no gap
df['is_gap']    = df['time_diff'] > GAP_THRESH_SEC

n_gaps = int(df['is_gap'].sum())
print(f'  {n_gaps:,} rows start a gap (> {GAP_THRESH_SEC}s since previous reading)')

Marking per-container time gaps ...
  54 rows start a gap (> 30s since previous reading)


## Step 3 — Scope anomaly labels to detectable fault types

This module's only inputs are container **CPU/memory** metrics (no network/trace
data). `loss` (packet loss) and `delay` (network latency) faults leave no signal in
these metrics at all — they were injected at the network layer. Keeping them
labeled `anomaly=1` would score the model against ground truth it structurally
cannot observe, deflating recall/F1 for no informative reason.

**Decision:** relabel `delay`/`loss` rows to normal (`label=0`, `failure_type=None`)
across all 4 cases. Rows are *not* dropped (that would fabricate a timestamp gap in
otherwise-real data) — only the label is corrected to match what's actually
observable. Detectable scope becomes: `cpu`, `memory`, `pod-failure`.

In [5]:
print('Scoping anomaly labels to detectable fault types ...')
before_counts = df['failure_type'].value_counts(dropna=False)

excl_mask = df['failure_type'].isin(EXCLUDED_FAULT_TYPES)
print(f'  Rows relabeled to normal: {excl_mask.sum():,}')
df.loc[excl_mask, 'label']        = 0
df.loc[excl_mask, 'failure_type'] = None

after_counts = df['failure_type'].value_counts(dropna=False)
comp = pd.DataFrame({'before': before_counts, 'after': after_counts}).fillna(0).astype(int)
print()
print('Failure-type distribution BEFORE -> AFTER:')
print(comp)

Scoping anomaly labels to detectable fault types ...
  Rows relabeled to normal: 704

Failure-type distribution BEFORE -> AFTER:
              before   after
failure_type                
cpu              244     244
delay            204       0
loss             500       0
memory           356     356
pod-failure      160     160
NaN           399432  399432
None               0     704


## Step 4 — Save the full cleaned dataset

All 4 cases, deduplicated, gap-marked, rescoped — kept as a reference/debugging
artifact. Not consumed directly by training (the split files below are).

In [6]:
print('Saving full cleaned dataset (all 4 cases) ...')
cleaned_path = os.path.join(OUT_DIR, 'all_cases_cleaned.csv')
df.to_csv(cleaned_path, index=False)
size_mb = os.path.getsize(cleaned_path) / (1024 * 1024)
print(f'  Saved: {cleaned_path}  ({len(df):,} rows, {size_mb:.1f} MB)')
print()
print('Label distribution per case (final, cleaned):')
for case, g in df.groupby('case'):
    types = g.loc[g.label == 1, 'failure_type'].value_counts().to_dict()
    print(f'  {case:15s} rows={len(g):>7,}  anomalies={g["label"].sum():>5,}  '
          f'({g["label"].mean()*100:.3f}%)  types={types}')

Saving full cleaned dataset (all 4 cases) ...
  Saved: c:\Users\jthar\Documents\Claude\Projects\module3\data\processed\all_cases_cleaned.csv  (400,896 rows, 71.8 MB)

Label distribution per case (final, cleaned):
  complex_case1   rows=223,803  anomalies=  256  (0.114%)  types={'pod-failure': 92, 'cpu': 88, 'memory': 76}
  complex_case2   rows= 77,760  anomalies=  372  (0.478%)  types={'memory': 240, 'cpu': 120, 'pod-failure': 12}
  single_case1    rows= 60,480  anomalies=   76  (0.126%)  types={'memory': 40, 'cpu': 36}
  single_case2    rows= 38,853  anomalies=   56  (0.144%)  types={'pod-failure': 56}


## Step 5 — CC1-only split + drift-evaluation sets

Per the agreed design: the VAE trains/validates/tests **exclusively on
complex_case1**. `single_case1`, `single_case2`, `complex_case2` are held out
entirely and evaluated separately afterward for generalization/drift — never
merged into the training distribution.

Split is by **global timestamp** (not row-count), so all containers share the same
train/val/test time boundary — no container has data straddling two splits at the
same moment. All CC1 anomalies route to `test` regardless of when they occurred
(verified below: none land at/after the val→test cutoff anyway, so there's no
double-counting).

In [7]:
print('Splitting complex_case1 by time (train/val/test) ...')
cc1     = df[df['case'] == 'complex_case1'].copy()
normals = cc1[cc1['label'] == 0]

t1 = normals['timestamp'].quantile(TRAIN_FRAC)
t2 = normals['timestamp'].quantile(VAL_FRAC)
print(f'  70th pct cutoff (train/val) : {pd.to_datetime(t1, unit="s")}')
print(f'  80th pct cutoff (val/test)  : {pd.to_datetime(t2, unit="s")}')

cc1_train = normals[normals['timestamp'] < t1].copy()
cc1_val   = normals[(normals['timestamp'] >= t1) & (normals['timestamp'] < t2)].copy()
cc1_test  = pd.concat([
    normals[normals['timestamp'] >= t2],
    cc1[cc1['label'] == 1],                 # ALL CC1 anomalies go to test, regardless of timestamp
]).sort_values(['cmdb_id', 'timestamp']).copy()

late_anomalies = int((cc1.loc[cc1['label'] == 1, 'timestamp'] >= t2).sum())
print(f'  (anomaly rows already at/after test cutoff: {late_anomalies} — no double-count risk)')

print()
print(f'  cc1_train : {len(cc1_train):,} rows (all normal)')
print(f'  cc1_val   : {len(cc1_val):,} rows (all normal)')
print(f'  cc1_test  : {len(cc1_test):,} rows  ({(cc1_test["label"]==1).sum():,} anomalies, '
      f'{(cc1_test["label"]==1).mean()*100:.2f}%)')

# Drift-evaluation sets: entire cleaned case, kept intact, reported separately (never merged)
drift_sets = {
    'single_case1':  df[df['case'] == 'single_case1'].copy(),
    'single_case2':  df[df['case'] == 'single_case2'].copy(),
    'complex_case2': df[df['case'] == 'complex_case2'].copy(),
}
print()
for name, d in drift_sets.items():
    print(f'  drift[{name}]: {len(d):,} rows  ({d["label"].sum():,} anomalies, {d["label"].mean()*100:.2f}%)')

Splitting complex_case1 by time (train/val/test) ...
  70th pct cutoff (train/val) : 2024-06-25 05:39:00
  80th pct cutoff (val/test)  : 2024-06-25 09:06:00
  (anomaly rows already at/after test cutoff: 0 — no double-count risk)

  cc1_train : 156,479 rows (all normal)
  cc1_val   : 22,356 rows (all normal)
  cc1_test  : 44,968 rows  (256 anomalies, 0.57%)

  drift[single_case1]: 60,480 rows  (76 anomalies, 0.13%)
  drift[single_case2]: 38,853 rows  (56 anomalies, 0.14%)
  drift[complex_case2]: 77,760 rows  (372 anomalies, 0.48%)


## Step 6 — Fit scaling on CC1-train only

`RobustScaler` (median/IQR), fit **only** on `cc1_train` — no other split or case
contributes to the scaling parameters, avoiding any leakage.

One verified data quirk: the 3 CPU-rate features are exactly 0 at both the 25th and
75th percentile in CC1-train (containers are idle most 15s windows), giving them an
IQR of 0. This was checked empirically — `RobustScaler` already guards against
divide-by-zero here (falls back to `scale_=1` for those columns). A `±20` clip is
still applied as a defensive margin, since an equivalent near-zero-spread issue
caused a real runaway-scale bug earlier in this project's history (PCA-space
feature, prior VAE iteration).

In [8]:
print('Fitting RobustScaler on complex_case1 TRAIN normals only ...')
scaler = RobustScaler()
scaler.fit(cc1_train[FEATURE_COLS])

print('  center_ (median):', np.round(scaler.center_, 4))
print('  scale_  (IQR)   :', np.round(scaler.scale_, 4))

def scale_and_clip(frame):
    out = frame.copy()
    out[FEATURE_COLS] = np.clip(scaler.transform(out[FEATURE_COLS]), -CLIP, CLIP)
    return out

cc1_train = scale_and_clip(cc1_train)
cc1_val   = scale_and_clip(cc1_val)
cc1_test  = scale_and_clip(cc1_test)
drift_sets = {name: scale_and_clip(d) for name, d in drift_sets.items()}

print()
print('Scaled cc1_train feature ranges:')
print(cc1_train[FEATURE_COLS].agg(['min', 'mean', 'max']).round(3))

Fitting RobustScaler on complex_case1 TRAIN normals only ...
  center_ (median): [0.     0.     0.     0.7934 0.7924 0.762  0.4545]
  scale_  (IQR)   : [1.     1.     1.     0.4476 0.4528 0.4531 0.4484]

Scaled cc1_train feature ranges:
      container_cpu_usage_seconds_rate  container_cpu_system_seconds_rate  container_cpu_user_seconds_rate  container_memory_usage_bytes  container_memory_working_set_bytes  container_memory_rss  \
min                              0.000                              0.000                            0.000                        -1.772                              -1.750                -1.682   
mean                             0.087                              0.097                            0.072                        -0.293                              -0.267                -0.170   
max                              1.000                              1.000                            1.000                         0.461                               0.

## Step 7 — Save split files + scaler

In [9]:
print('Saving split files ...')

SAVE_COLS = ['timestamp', 'cmdb_id', 'case'] + FEATURE_COLS + ['label', 'failure_type', 'time_diff', 'is_gap']

splits = {
    'cc1_train': cc1_train,
    'cc1_val':   cc1_val,
    'cc1_test':  cc1_test,
    'drift_single_case1':  drift_sets['single_case1'],
    'drift_single_case2':  drift_sets['single_case2'],
    'drift_complex_case2': drift_sets['complex_case2'],
}

for name, d in splits.items():
    path = os.path.join(OUT_DIR, f'{name}.csv')
    d[SAVE_COLS].to_csv(path, index=False)
    print(f'  {name:20s} -> {path}  ({len(d):,} rows)')

scaler_path = os.path.join(MODEL_DIR, 'cc1_scaler.pkl')
joblib.dump({'scaler': scaler, 'feature_cols': FEATURE_COLS, 'clip': CLIP}, scaler_path)
print(f'\n  Scaler saved -> {scaler_path}  (reuse in the windowing/training notebook — do not refit)')

Saving split files ...
  cc1_train            -> c:\Users\jthar\Documents\Claude\Projects\module3\data\processed\cc1_train.csv  (156,479 rows)
  cc1_val              -> c:\Users\jthar\Documents\Claude\Projects\module3\data\processed\cc1_val.csv  (22,356 rows)
  cc1_test             -> c:\Users\jthar\Documents\Claude\Projects\module3\data\processed\cc1_test.csv  (44,968 rows)
  drift_single_case1   -> c:\Users\jthar\Documents\Claude\Projects\module3\data\processed\drift_single_case1.csv  (60,480 rows)
  drift_single_case2   -> c:\Users\jthar\Documents\Claude\Projects\module3\data\processed\drift_single_case2.csv  (38,853 rows)
  drift_complex_case2  -> c:\Users\jthar\Documents\Claude\Projects\module3\data\processed\drift_complex_case2.csv  (77,760 rows)

  Scaler saved -> c:\Users\jthar\Documents\Claude\Projects\module3\models\cc1_scaler.pkl  (reuse in the windowing/training notebook — do not refit)


## Step 8 — Final verification

In [10]:
print('=== FINAL VERIFICATION ===\n')
for name in ['all_cases_cleaned', 'cc1_train', 'cc1_val', 'cc1_test',
             'drift_single_case1', 'drift_single_case2', 'drift_complex_case2']:
    path = os.path.join(OUT_DIR, f'{name}.csv')
    d = pd.read_csv(path, low_memory=False)
    n_anom = int(d['label'].sum())
    size_mb = os.path.getsize(path) / 1e6
    print(f'{name:22s} rows={len(d):>7,}  anomalies={n_anom:>5,}  '
          f'({n_anom / len(d) * 100:.3f}%)  size={size_mb:.1f}MB')

=== FINAL VERIFICATION ===

all_cases_cleaned      rows=400,896  anomalies=  760  (0.190%)  size=75.3MB
cc1_train              rows=156,479  anomalies=    0  (0.000%)  size=26.3MB
cc1_val                rows= 22,356  anomalies=    0  (0.000%)  size=3.8MB
cc1_test               rows= 44,968  anomalies=  256  (0.569%)  size=7.6MB
drift_single_case1     rows= 60,480  anomalies=   76  (0.126%)  size=10.1MB
drift_single_case2     rows= 38,853  anomalies=   56  (0.144%)  size=6.5MB
drift_complex_case2    rows= 77,760  anomalies=  372  (0.478%)  size=13.1MB


## Output files

| File | Contents | Purpose |
|---|---|---|
| `all_cases_cleaned.csv` | all 4 cases, deduped + gap-marked + rescoped, **unscaled** | reference/debugging only |
| `cc1_train.csv` | CC1 normals, first 70% by time, CC1-scaled | VAE trains here (unsupervised) |
| `cc1_val.csv` | CC1 normals, next 10%, CC1-scaled | early stopping / threshold pre-tuning |
| `cc1_test.csv` | CC1 normals (last 20%) + all CC1 anomalies, CC1-scaled | in-distribution evaluation |
| `drift_single_case1.csv` | full single_case1, CC1-scaled | drift-eval set 1 |
| `drift_single_case2.csv` | full single_case2, CC1-scaled | drift-eval set 2 |
| `drift_complex_case2.csv` | full complex_case2, CC1-scaled | drift-eval set 3 (only genuinely distinct deployment) |
| `models/cc1_scaler.pkl` | fitted `RobustScaler` + config | reuse downstream, never refit |

**Deliberately not done here:** sliding-window construction, flattening, PCA,
VAE training/eval — that's the next notebook, built on these split files.